# Owner and Transaction Scale Calculations
Created by Nicholas Polimeni

Updated by Melissa Juarez to include more data cleaning

In [1]:
import pandas as pd

pd.set_option('display.max_columns', 150)
pd.options.display.float_format = '{:,.3f}'.format

In [2]:
# Cleaned digest and sales data from clean_data.ipynb
FILES_PATH = 'data/'
digest_full = pd.read_csv(
    FILES_PATH + 'fulton_ALL_ownership_2011_2022.csv',
    dtype={
        "taxyr": 'Int64',
        "parid": str,
        "nbhd": str,
        "situs_adrno": 'Int64',
        "situs_adrdir": str,
        "situs_adrstr": str,
        "situs_adrsuf": str,
        "situs_adrsuf2": str,
        "situs_cityname": str,
        "class": str,
        "luc": str,
        "livunit": 'Int64',
        "calcacres": float,
        "ofcard": 'Int64',
        "taxdist": str,
        "own1": str,
        "own2": str,
        "owner_adrno": 'Int64',
        "owner_adrdir": str,
        "owner_adrstr": str,
        "owner_adrsuf": str,
        "owner_adrsuf2": str,
        "owner_cityname": str,
        "statecode": str,
        "country": str,
        "unitno": str,
        "zip1": str,
        "aprtot": float,
        "revcode": 'Int64',
        "revreas": str,
        "revtot": str,
        "yrblt": 'Int64',
        "sf": 'Int64',
        "grade": str,
        "note1": str,
        "note2": str
    }
)
sales_full = pd.read_csv(FILES_PATH + '1727793735_fulton_sales.csv')

## Modify data to enable aggreggating on entity key

In [3]:
## goal: reduce number of property adrnos that are 0 by accounting for the following data error cases
digest_full['mod_own_adrstr'] = digest_full['owner_adrstr'].copy(deep=True)
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].str.replace(r'[.?]', '', regex=True) ## remove ?s and periods
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].replace(r'^(?:X+)?$', pd.NA, regex=True) ## remove where cell is "" or only Xs


### Identify same owners in parcel data
- Drop any rows without Owner Address
- Create an Owner Address (labeled: "owner_addr") column that is the concatentation of owner address number, owner address string, and owner zip.
- If address string contains numbers, then it is a PO BOX. However, a lot are formatted in different ways, such as P O BOX 123, PO BOX 123, P.O. BOX 123, etc. We can retain the number from the address string, and manually prepend PO BOX, so all will have an identical format.

**Why:** these values get us a highly accurate key for same owner. Owner address string does not contain postfixes like ST, AVE, etc. that might cause issues. Combined with owner number and owner zip, we can say with high confidence that the address is the same while avoiding many common differences amongst the same address (ST vs STREET, etc.). This method is prefered over names which has a higher chance of false positive, and large corporations may operate with differently named subsidaries. This method may also undercount, if a company uses multiple addresses, but this is somewhat unlikely and undercounting is simply an acceptable limitation. It is acceptable since large investors (who would use different addresses) will own so many properties with each subsidary that it will be binned in the correct bin regardless.

In [4]:
# Drop rows w/o owner address; drop those where address is only Xs or ?s
digest_full = digest_full.dropna(subset=["owner_adrstr"])

# Re-format PO BOXES
re_box_and_numbers = r".*BOX.*[0-9].*"
re_capture_numbers = r"([0-9]+)"

mask = digest_full["mod_own_adrstr"].str.contains(re_box_and_numbers, regex=True, na=False)

digest_full.loc[mask, "mod_own_adrstr"] = "PO BOX " + digest_full.loc[
    mask, "mod_own_adrstr"
].str.extract(re_capture_numbers)[0]

# clean unit numbers remove #s & extra spaces
digest_full['mod_unitno'] = digest_full['unitno'].copy(deep=True)
digest_full['mod_unitno'] = digest_full['mod_unitno'].str.replace(r'[#-]', '', regex=True) ## remove hashtags & hyphens
digest_full['mod_unitno'] = digest_full['mod_unitno'].str.replace(r"\s{1,}", '', regex=True) ## remove spaces
digest_full['mod_unitno'] = digest_full['mod_unitno'].fillna('')  # missing value with ""

In [5]:
mask.sum()

np.int64(176077)

In [6]:
# Print total number of PO BOXES without a number in their address string
re_po_box_no_number = r"^(?!.*\d)[P]+.* BOX.*"
len(digest_full[digest_full["mod_own_adrstr"].str.contains(
    re_po_box_no_number, regex=True, na=False
)][["owner_adrno", "mod_own_adrstr"]])

105

In [7]:
# Regex to clean by replacing dots, commas, and multiple spaces
# Also make all strings uppercase (they should be already)

re_dots_commas = r"[.,]+"
re_multiple_spaces = r"\s{2,}"

digest_full["owner_addr"] = (
    digest_full["owner_adrno"].astype(str) + " " +
    digest_full["mod_own_adrstr"] + " " +
    digest_full["mod_unitno"] + " " +
    digest_full["zip1"].astype(str)
).str.replace(
    re_dots_commas,
    "",
    regex=True
).str.replace(
    re_multiple_spaces,
    " ",
    regex=True
).str.upper()

In [8]:
digest_full[
    ["owner_adrno", "owner_adrstr", "mod_own_adrstr", "unitno", "mod_unitno", "zip1", "owner_addr"]
].sample(10)

,owner_adrno,owner_adrstr,mod_own_adrstr,unitno,mod_unitno,zip1,owner_addr
637578,375,OVERHILL,OVERHILL,NaN,,30005,375 OVERHILL 30005
593418,96,THE PRADO,THE PRADO,NaN,,30309,96 THE PRADO 30309
2739084,355,WYNLAND,WYNLAND,NaN,,30350,355 WYNLAND 30350
62481,8225,MC GINNIS FERRY,MC GINNIS FERRY,NaN,,30174,8225 MC GINNIS FERRY 30174
2671392,955,JUNIPER,JUNIPER,1118,1118,30309,955 JUNIPER 1118 30309
4168764,1035,BETHANY CREEK,BETHANY CREEK,NaN,,30004,1035 BETHANY CREEK 30004
2575988,3618,JEFFERSON,JEFFERSON,NaN,,30337,3618 JEFFERSON 30337
4416337,3595,RENAISSANCE,RENAISSANCE,NaN,,30349,3595 RENAISSANCE 30349
833774,<NA>,P.O. BOX 1831,PO BOX 1831,NaN,,30298,<NA> PO BOX 1831 30298
2481795,9845,SUMMER OAKS,SUMMER OAKS,NaN,,30076,9845 SUMMER OAKS 30076


### Identify same owners in sales data

**Method**:
- Sales data does not contain buyer or seller address. We can't simply use GRANTEE or GRANTOR name, because names can be different for the same owner corporation (subsidaries, typos). Instead we identify buyer and seller address by:
    - Match GRANTEE name to parcel data on GRANTEE = Own1 (owner name) and extract owner_addr for CURRENT TAXYR
    - Match GRANTOR name to parcel data on GRANTOR = Own1 (owner name) and extract owner_addr for PREVIOUS TAXYR
    - For names where the GRANTEE or GRANTOR name doesn't match exactly (due to typos, etc.), we can take the owner_addr with the same method ONLY IF there was only one sale in the given TAXYR. In the case of multiple sales in one TAXYR, the last purchaser appears to be recorded in the parcel data as the owner (see evidence below); if we tried to match an earlier sale in that year, we would get the wrong owner address. This is a problem because we want the purchaser address for each sale to appropriately account for flipping activity for example.
    - Else, try to find an exact owner name match from all parcel data, not limited to PARID and TAXYR; use the last match if a match is found (last because that is most recent address of company).
- In short:
    - Try to match by owner name, PARID, and TAXYR
    - If no match, get match from just PARID and TAXYR, ONLY IF there is a single transcation in the given TAXYR for that PARID
    - Else, try to find an exact owner name match from all parcel data, not limited to PARID and TAXYR; use the last match if a match is found.
    - Where none of the above methods work, drop if total count is insignificant

In [9]:
# Minor cleaning on GRANTEE, GRANTOR, and Own1 (parcel data)
# Regex to clean by replacing dots, commas, and multiple spaces
# Also make all strings uppercase (they should be already)
re_dots_commas = r"[.,]+"
re_multiple_spaces = r"\s{2,}"

for col in ["grantee", "grantor"]:
    sales_full[col] = sales_full[col].str.replace(
        re_dots_commas, "", regex=True
    ).str.replace(
        re_multiple_spaces, " ", regex=True
    ).str.upper()
    
digest_full["own1"] = digest_full["own1"].str.replace(
    re_dots_commas, "", regex=True
).str.replace(
    re_multiple_spaces, " ", regex=True
).str.upper()

In [10]:
### number of sales per parcel-tax year combination
count_sales_yr = pd.DataFrame(
    sales_full.groupby(["tax_year", "parcel_id"])["parcel_id"].count()
).rename(columns={"parcel_id": "count_sales_yr"})

## merge counts into original sales dataset
sales_full = sales_full.merge(
    count_sales_yr,
    on=["tax_year", "parcel_id"],
    how="inner"
)

## num tax-parcel combos sold more than once that year 
more_than_one_sale_yr = len(
    sales_full[sales_full["count_sales_yr"] > 1].drop_duplicates(
        subset=["tax_year", "parcel_id"]
    )
)

## num of tax parcel combos sold more than once w a valid sale
more_than_one_sale_yr_valid = len(
    sales_full[
       (sales_full["count_sales_yr"] > 1)
       & (sales_full["sale_val"] == "0")
    ].drop_duplicates(
        subset=["tax_year", "parcel_id"]
    )
)

print(f"Count of properties that sold multiple times in one year: {more_than_one_sale_yr}")
print(f"Count of properties that sold multiple times in one year (valid only): {more_than_one_sale_yr_valid}")
count_sales_yr.sort_values(by="count_sales_yr", ascending=False).head(5)

Count of properties that sold multiple times in one year: 42251
Count of properties that sold multiple times in one year (valid only): 15609


count_sales_yr
tax_year parcel_id                      
2022     14 007800070969              36
2020     14 008900010095              17
2022     14 005700100241              16
2019     14 008200010308              15
         14 007400040032              14

In [11]:
print(len(sales_full))
for person in ["grantee", "grantor"]:
    # grantee == buyer, grantor == seller

    # create copy of digest data + rename columns to be consistent
    digest_df = digest_full[['parid', 'taxyr', 'owner_addr', 'own1']].rename(
        columns={"parid": "parcel_id", "taxyr": "tax_year"}
    ).copy(deep=True)

    # subset sales to include tax yr, parcel id, grantee/grantor, and num of sales that year
    sale_df = sales_full[["tax_year", "parcel_id", f"{person}", "count_sales_yr"]].copy(deep=True)
    
    # if seller, shift tax yr in digest up by one. this is because we want to find the seller's address data
    # in the digests. if they sold the prop in 2021 (sales data), then they will be the owner in 2020 (digest). 
    # to match up the digest and sales years, we need to add 1 year to the digest tax yr.
    if person == "grantor":
        digest_df["tax_year"] = digest_df["tax_year"] + 1

    # match tax yr, parcel, and buyer/seller from sales data to tax yr, parcel, and owner in digest 
    exact_match = sale_df.merge(
        digest_df,
        left_on=["parcel_id", "tax_year", f"{person}"],
        right_on=["parcel_id", "tax_year", "own1"],
    ).rename(
        columns={"own1": f"{person}_exact", "owner_addr": f"{person}_exact_addr"}
    ).drop_duplicates(subset=["parcel_id", "tax_year"])

    # merge just on exact names
    exact_name_match = sale_df.merge(
        digest_df[["own1", "owner_addr"]].drop_duplicates(subset="own1", keep="last"),
        left_on=[f"{person}"],
        right_on=["own1"],
    ).rename(columns={
        "own1": f"{person}_only_exact_name", "owner_addr": f"{person}_only_exact_name_addr"
    }).drop_duplicates(subset=[f"{person}_only_exact_name"])

    # merge when name could be diff but there is only one sale in a given tax year
    single_sale_match = sale_df[
        sale_df["count_sales_yr"] < 2
    ].merge(
        digest_df,
        left_on=["parcel_id", "tax_year"],
        right_on=["parcel_id", "tax_year"],
    ).rename(
        columns={"own1": f"{person}_single_sale", "owner_addr": f"{person}_single_sale_addr"}
    ).drop_duplicates(subset=["parcel_id", "tax_year"])
    
    ## add match data to full sales dataset
    sales_full = sales_full.merge(
        exact_match[["tax_year", "parcel_id", f"{person}_exact", f"{person}_exact_addr"]],
        left_on=["tax_year", "parcel_id", f"{person}"],
        right_on=["tax_year", "parcel_id", f"{person}_exact"],
        how="left"
    )
    
    sales_full = sales_full.merge(
        single_sale_match[["tax_year", "parcel_id", f"{person}_single_sale", f"{person}_single_sale_addr"]],
        on=["tax_year", "parcel_id"],
        how="left"
    )
    
    sales_full = sales_full.merge(
        exact_name_match[[f"{person}_only_exact_name", f"{person}_only_exact_name_addr"]],
        left_on=[f"{person}"],
        right_on=[f"{person}_only_exact_name"],
        how="left"
    )

len(sales_full)

427617


427617

In [12]:
for person in ["grantee", "grantor"]:
    print(f"Person: {person} ---")
    sales_full[f"{person}_match"] = sales_full[f"{person}_exact"]
    sales_full[f"{person}_match_addr"] = sales_full[f"{person}_exact_addr"]
    num_matched = len(sales_full[sales_full[f'{person}_match'].notna()])
    print(f"Number exact matched: {num_matched}")
    print(f"Pct exact matched: {num_matched / len(sales_full)}")
    print("")
    
    for match in ["single_sale", "only_exact_name"]:
        sales_full[f"{person}_match"] = sales_full[f"{person}_match"].fillna(
            sales_full[f"{person}_{match}"]
        )
        sales_full[f"{person}_match_addr"] = sales_full[f"{person}_match_addr"].fillna(
            sales_full[f"{person}_{match}_addr"]
        )
        prev_matched = num_matched
        num_matched = len(sales_full[sales_full[f'{person}_match'].notna()])
        print(f"Number of additional matches with {match}: {num_matched - prev_matched}")
        print(f"Number prev matches + {match} matched: {num_matched}")
        print(f"Pct prev matches + {match} matched: {num_matched / len(sales_full)}")
        print("")
    
    print("")
    print("")

Person: grantee ---
Number exact matched: 343932
Pct exact matched: 0.8042991742610794

Number of additional matches with single_sale: 32317
Number prev matches + single_sale matched: 376249
Pct prev matches + single_sale matched: 0.8798738123133552

Number of additional matches with only_exact_name: 33128
Number prev matches + only_exact_name matched: 409377
Pct prev matches + only_exact_name matched: 0.9573450073313269



Person: grantor ---
Number exact matched: 171872
Pct exact matched: 0.4019297642516551

Number of additional matches with single_sale: 172993
Number prev matches + single_sale matched: 344865
Pct prev matches + single_sale matched: 0.8064810332610725

Number of additional matches with only_exact_name: 41258
Number prev matches + only_exact_name matched: 386123
Pct prev matches + only_exact_name matched: 0.902964568761298





Save a random sample of 100 to manually verify

In [13]:
sales_full.sample(100)[
    ["tax_year", "parcel_id", "grantee", "grantee_match", "grantor", "grantor_match"]
].sort_values(by="tax_year").to_csv('output/sales_matches_sample.csv', index=False)

## Drop govt and bank entities from transcation data
- Retain a dataset where these were only dropped from GRANTEE, because someone buying from one of these entities is still influencing the market and thus this should be included in their transcation scale

In [16]:
govt_keywords = ['FEDERAL', 'SECRETARY', 'CITY', 'COUNTY', 'FANNIE', 'FREDDIE']
bank_keywords = [
    'BANK', 'MORTGAGE', 'LENDING', 'LOAN',
    'FINANCE', 'FUND', 'CREDIT', 'TRUST'
]
govt = []
banks = []

govt += sales_full[
    sales_full['grantee'].apply(lambda x: any([key in str(x) for key in govt_keywords]))
]['grantee'].unique().tolist() + sales_full[
    sales_full['grantor'].apply(lambda x: any([key in str(x) for key in govt_keywords]))
]['grantor'].unique().tolist() + digest_full[
    digest_full["own1"].apply(lambda x: any([key in str(x) for key in govt_keywords]))
]['own1'].unique().tolist() + digest_full[
    digest_full["own2"].apply(lambda x: any([key in str(x) for key in govt_keywords]))
]['own2'].unique().tolist()

banks += sales_full[
    sales_full['grantee'].apply(lambda x: any([key in str(x) for key in bank_keywords]))
]['grantee'].unique().tolist() + sales_full[
    sales_full['grantor'].apply(lambda x: any([key in str(x) for key in bank_keywords]))
]['grantor'].unique().tolist() + digest_full[
    digest_full["own1"].apply(lambda x: any([key in str(x) for key in bank_keywords]))
]['own1'].unique().tolist() + digest_full[
    digest_full["own2"].apply(lambda x: any([key in str(x) for key in bank_keywords]))
]['own2'].unique().tolist()

# Save record of entities classified as govt or bank
with open("./output/govt.txt", "w") as f:
    f.write("\n".join(govt))
with open("./output/banks.txt", "w") as f:
    f.write("\n".join(banks))

In [17]:
init_len_sales = len(sales_full)
init_len_digest = len(digest_full)

sales_grantee_dropped = sales_full[
    ~(sales_full["grantee"].isin(govt + banks))
].copy(deep=True)

sales_full = sales_full[
    ~((sales_full["grantee"].isin(govt + banks))
    | (sales_full["grantor"].isin(govt + banks)))
]

print("Number dropped from sales: ", init_len_sales - len(sales_full))

Number dropped from sales:  0


## Identify corporate owners, create corp owner flags for each record
- grantee, grantor in sales
- own1 in digest

In [19]:
# Any with risk of false positive like "CO" need to have a space prepended or postpended
corp_keywords = [
    'LLC', ' INC', 'LLP', 'L.L.C', 'L.L.P', 'I.N.C', 'L L C',
    'L L P', ' L P', ' LP', 'LTD', ' CORP', 'CORPORATION',
    'COMPANY', ' CO ', 'LIMITED', 'PARTNERSHIP', 'PARTNERSHIPS',
    'ASSOCIATION', 'ASSOC', 'INCORPORATED', 'INCORP',
    'L.T.D', 'LTD', "HOME", "SOLUTIONS"
]

# Make a list of all corp owners -- added own2 as well.
corps = sales_full[
    sales_full['grantee'].apply(lambda x: any([key in str(x) for key in corp_keywords]))
]['grantee'].unique().tolist() + sales_full[
    sales_full['grantor'].apply(lambda x: any([key in str(x) for key in corp_keywords]))
]['grantor'].unique().tolist() + digest_full[
    digest_full["own1"].apply(lambda x: any([key in str(x) for key in corp_keywords]))
]['own1'].unique().tolist() + digest_full[
    digest_full["own2"].apply(lambda x: any([key in str(x) for key in corp_keywords]))
]['own2'].unique().tolist()

with open("./output/corp_names.txt", "w") as f:
    f.write("\n".join(corps))

In [ ]:
# Flag for any corp owner
sales_full["grantee_corp_flag"] = sales_full['grantee'].isin(corps).astype(int)
sales_full["grantor_corp_flag"] = sales_full['grantor'].isin(corps).astype(int)

digest_full["own_corp_flag"] = (
    digest_full["own1"].isin(corps) | digest_full["own2"].isin(corps)
).astype(int)

In [21]:
sales_full[['grantee', 'grantee_corp_flag', 'grantor', 'grantor_corp_flag']].sample(10)

,grantee,grantee_corp_flag,grantor,grantor_corp_flag
230605,RAY KEAN C,0,D R HORTON INC,1
238325,FRIEDMAN HOPE S,0,FRIEDMAN GEOFFREY T,0
201770,SAFE FUTURE REAL ESTATE INVESTMENTS LLC,1,CARLTON POINTE HOLDING COMPANY LLC,1
334094,MEEKINS ANDREW T,0,CENTURY COMMUNITIES OF GEORGIA LLC,1
69284,CORNELLA ROBERT,0,PROVIDENCE GROUP OF GEORGIA CUSTOM HOMES,1
8373,VENTURE HOMES INC,1,RL REGI GEORGIA LLC A GEORGIA LIMITED L,1
412447,FINCH TRACI,0,OVERTON RANDOLPH,0
15929,CHAPPELL FOREST HOLDING PARTNERS INC,1,SE APARTMENT HOLDING GROUP LLC,1
220149,PENICK SUSAN,0,RASAR ELIZABETH,0
254246,WILKINS FRANCES GREENE,0,ESTATE OF WILLIAM EARL WILKINS DECEASED,0


In [30]:
digest_full[['own1', 'own2', 'own_corp_flag']].sample(10)

,own1,own2,own_corp_flag
536156,DUTCH REAL ESTATE INVESTORS,LLC,1
4348878,HUNTER PATRICIA ANN,NaN,0
4114944,COOPER JEANNETTE R,NaN,0
1553458,OMER KEDER &,ABDULKADIR ZALIKA H,0
1075777,JOHNS GUY,NaN,0
1219252,RENDER MATTIE ET AL,NaN,0
2001695,ROBINSON HARRY E JR & MARCIA P,NaN,0
1930829,SMITH JULIETTA E,NaN,0
470899,CIRSA USA LLC,NaN,1
1385369,SECRETARY OF HOUSING & URBAN DEVELOPMENT,NaN,0


## Create a rental property flag

In [31]:
# when owner address is not the same as property address
digest_full["rental_flag"] = 0
digest_full.loc[
    ((digest_full["situs_adrno"] != digest_full["owner_adrno"])
    & (digest_full["situs_adrstr"] != digest_full["owner_adrstr"])),
    "rental_flag"
] = 1

In [32]:
digest_full[['situs_adrno', 'owner_adrno', 'situs_adrstr', 'owner_adrstr', 'rental_flag']].sample(20)

,situs_adrno,owner_adrno,situs_adrstr,owner_adrstr,rental_flag
1015107,6325,6325,CEDAR GROVE,CEDAR GROVE,0
1655979,5310,5310,CROSS ROADS MANOR,CROSS ROADS,0
2168585,1965,2774,VELMA,COBB,1
3429379,12792,12792,WATERSIDE,WATERSIDE,0
1678266,335,335,WINN PARK,WINN PARK,0
2009690,1378,273,WESLEY OAKS,12TH,1
2075335,6423,6449,GRESHAM,ROOSEVELT,1
3501089,1060,1060,TIMBERLINE,TIMBERLINE,0
104212,5,5,PEPPERMILL,PEPPERMILL,0
2275859,1201,1201,PINE HEIGHTS,PINE HEIGHTS,0


## For each sale, create a dummy variable for each sale type
- corp purchase from ind
- corp sale to ind
- ind to ind
- corp to corp

In [33]:
# Sale type matrix

sales_full['corp_bought_ind'] = 0
sales_full['corp_sold_ind'] = 0
sales_full['ind_to_ind'] = 0
sales_full['corp_to_corp'] = 0

sales_full.loc[
    (sales_full["grantee_corp_flag"] == 1) & (sales_full["grantor_corp_flag"] == 0), 'corp_bought_ind'
] = 1
sales_full.loc[
    (sales_full["grantee_corp_flag"] == 0) & (sales_full["grantor_corp_flag"] == 0), 'ind_to_ind'
] = 1
sales_full.loc[
    (sales_full["grantee_corp_flag"] == 0) & (sales_full["grantor_corp_flag"] == 1), 'corp_sold_ind'
] = 1
sales_full.loc[
    (sales_full["grantee_corp_flag"] == 1) & (sales_full["grantor_corp_flag"] == 1), 'corp_to_corp'
] = 1

# Validate sale matrix is correct
sales_full[[
    "grantee", "grantee_corp_flag", "grantor", "grantor_corp_flag", "corp_bought_ind", "ind_to_ind",
    "corp_sold_ind", "corp_to_corp"
]].sample(10)

,grantee,grantee_corp_flag,grantor,grantor_corp_flag,corp_bought_ind,ind_to_ind,corp_sold_ind,corp_to_corp
35254,ISABELLA PROPERTIES LLC,1,STRAIGHT ARROW DEVELOPMENT INC,1,0,0,0,1
351602,FERRER ALEXANDER,0,HICKS THOMAS W,0,0,1,0,0
272732,MANOHAR NIVEDH HARSHAN,0,BROCK BUILT HOMES LLC,1,0,0,1,0
346241,MONTONDO KEVIN,0,HAYWOOD CHRISTOPHER R,0,0,1,0,0
294290,KOLOTOS MICHAIL,0,DIXON NICOLE,0,0,1,0,0
128040,GREAT INVESTORS GROUP LLC,1,AKINWE ABIDEMI,0,1,0,0,0
419221,ELITE PINNACLE HOMES LLC,1,LOVELL ERNESTINE,0,1,0,0,0
103541,BOULEVARD APARTMENTS OWNER LP,1,MH PAULDING HOLDINGS LLC,1,0,0,0,1
300576,VERES SHAWN & LOVELL DANNY J,0,ALLIED PROPERTY GROUP LLC,1,0,0,1,0
196194,THOMPSON LATONYA MILLER,0,NCRC PRESTON HILLS LLC,1,0,0,1,0


## Create ownership scale table

In [34]:
owned_fulton_yr = pd.DataFrame(
    digest_full.groupby(["taxyr", "owner_addr"])["parid"].count()
).rename(columns={"parid": "count_owned_fulton_yr"}).reset_index()
owned_fulton_yr

assoc_owner_names = pd.DataFrame(
    digest_full.groupby(["owner_addr"]).agg({"own1": list})
).rename(columns={"own1": "assoc_owner_names"}).reset_index()

owner_scale = owned_fulton_yr.merge(
    assoc_owner_names,
    on=["owner_addr"],
    how="left"
)

owner_scale.sort_values(by="count_owned_fulton_yr", ascending=False).head(5)

,taxyr,owner_addr,count_owned_fulton_yr,assoc_owner_names
724955,2012,68 MITCHELL 1350 30303,1610,"[CITY OF ATLANTA, CITY OF ATLANTA, CITY OF ATL..."
986973,2013,68 MITCHELL 1350 30303,1609,"[CITY OF ATLANTA, CITY OF ATLANTA, CITY OF ATL..."
464203,2011,68 MITCHELL 1350 30303,1607,"[CITY OF ATLANTA, CITY OF ATLANTA, CITY OF ATL..."
1250013,2014,68 MITCHELL 1350 30303,1607,"[CITY OF ATLANTA, CITY OF ATLANTA, CITY OF ATL..."
1514784,2015,68 MITCHELL 1350 30303,1485,"[CITY OF ATLANTA, CITY OF ATLANTA, CITY OF ATL..."


In [35]:
owner_scale[
    owner_scale["taxyr"] == 2020
].sort_values(by="count_owned_fulton_yr", ascending=False).head(15)

,taxyr,owner_addr,count_owned_fulton_yr,assoc_owner_names
2862544,2020,68 MITCHELL 1350 30303,1414,"[CITY OF ATLANTA, CITY OF ATLANTA, CITY OF ATL..."
2682254,2020,141 PRYOR 7000 30303,644,"[FULTON COUNTY, FULTON COUNTY, FULTON COUNTY, ..."
2732450,2020,2424 PIEDMONT 30324,595,"[MARTA, MARTA, MARTA, MARTA, MARTA, MARTA, MAR..."
2822812,2020,5001 PLAZA ON THE 200 78746,549,"[JEFF 1 LLC, BTRA V LLC, RPA4 LLC, SFRA III LL..."
2697693,2020,1717 MAIN 2000 75201,495,"[2014 2 IH BORROWER L P, 2014 2 IH BORROWER L ..."
2744760,2020,270 WASHINGTON 30334,490,"[STATE OF GEORGIA, STATE OF GEORGIA, STATE OF ..."
2654673,2020,1110 NORTHCHASE 150 30067,424,"[NORTHWEST INTOWN DEVELOPMENT LLC, BROCK BUILT..."
2727410,2020,230 JOHN WESLEY DOBBS 30303,401,"[HOUSING AUTH CITY OF ATLANTA, HOUSING AUTH CI..."
2781277,2020,3505 KOGER BLVD 400 30096,389,"[FYR SFR BORROWER LLC, FYR SFR BORROWER LLC, F..."
2731962,2020,241 RALPH MCGILL 30308,376,"[GEORGIA POWER CO, GEORGIA POWER CO, GEORGIA P..."


In [61]:
""" owner_scale[
    (owner_scale["taxyr"] == 2020)
    & (owner_scale["owner_addr"] == "591 PUTNAM 6830")
].sort_values(by="count_owned_fulton_yr", ascending=False).head(15) """

' owner_scale[\n    (owner_scale["taxyr"] == 2020)\n    & (owner_scale["owner_addr"] == "591 PUTNAM 6830")\n].sort_values(by="count_owned_fulton_yr", ascending=False).head(15) '

### Identify major institutional investors

In [36]:
import re

owner_keywords = {
    "Amherst": ["AMHERST", "ARVM"],
    "Cerberus": ["CERBERUS", "FKH", "RM1 ", "RMI "],
    "Progress": ["PROGRESS", "FYR"],
    "Invitation": ["INVITATION", "IH "],
    "Colony": ["COLONY", "STARWOOD", "CSH", "CAH "],
    "Sylvan": ["SYLVAN", "RNTR"],
    "Tricon": ["TRICON", "TAH"]
}

for owner in owner_keywords:
    query_str = "|".join(owner_keywords[owner])

    filtered_rows = owner_scale[
        (owner_scale["taxyr"] == 2020) &
        owner_scale["assoc_owner_names"].apply(lambda x: any(
            ((re.search(query_str, name)) for name in x)
        ))
    ]
    
    filtered_rows = filtered_rows[
        filtered_rows["count_owned_fulton_yr"] > 49
    ]
    
    total = filtered_rows["count_owned_fulton_yr"].sum()
    print(f"{owner} owned {total} properties in 2020")
    display(filtered_rows)
    
    addresses = filtered_rows["owner_addr"].unique().tolist()
    print(addresses)
    
    names = [set(x) for x in filtered_rows[
        filtered_rows["count_owned_fulton_yr"] > 49
    ]["assoc_owner_names"].to_list()]

    names = set().union(*names)
    names = ", ".join(names)

    with open(f"./output/{owner}_names.txt", "w") as f:
        f.write("KEYWORDS: " + query_str + "\n")
        f.write("ADDRESSES: " + ", ".join(addresses) + "\n\n")
        f.write("NAMES\n--------------------\n")
        f.write(names)

Amherst owned 735 properties in 2020


,taxyr,owner_addr,count_owned_fulton_yr,assoc_owner_names
2822812,2020,5001 PLAZA ON THE 200 78746,549,"[JEFF 1 LLC, BTRA V LLC, RPA4 LLC, SFRA III LL..."
2822820,2020,5001 PLAZA ON THE LAKE 200 78746,93,"[HFS I ASSETS COMPANY LLC, HFS I ASSETS COMPAN..."
2888776,2020,8300 MOPAC EXPRESSWAY 200 78759,93,"[JEFF 1 LLC, JEFF 1 LLC, JEFF 1 LLC, JEFF I LL..."


['5001 PLAZA ON THE 200 78746', '5001 PLAZA ON THE LAKE 200 78746', '8300 MOPAC EXPRESSWAY 200 78759']
Cerberus owned 262 properties in 2020


,taxyr,owner_addr,count_owned_fulton_yr,assoc_owner_names
2703779,2020,1850 PARKWAY 900 30067,176,"[CSMA BLT LLC, CERBERUS SFR HOLDINGS L P, CERB..."
2910483,2020,<NA> PO BOX 2249 30028,86,"[WALTON GA WOODBURY PARK LP ET AL, ENCLAVE ATL..."


['1850 PARKWAY 900 30067', '<NA> PO BOX 2249 30028']
Progress owned 1002 properties in 2020


...

['3520 PIEDMONT 410 30305', '8665 HARTFORD 200 85255', '<NA> PO BOX 2458 30023']
Sylvan owned 711 properties in 2020


,taxyr,owner_addr,count_owned_fulton_yr,assoc_owner_names
2776458,2020,34 PEACHTREE 2200 30303,88,"[FULTON COUNTY CITY OF ATLANTA LAND, FULTON CO..."
2780486,2020,3495 PIEDMONT 11 30305,180,"[RNTR 3 LLC, RNTR 3 LLC, RNTR 1 LLC, RNTR 3 LL..."
2780496,2020,3495 PIEDMONT 300 30305,90,"[RNTR 3 LLC, RNTR 3 LLC, RNTR 3 LLC, RNTR 3 LL..."
2781275,2020,3505 KOGER 400 30096,133,"[RNTR 3 LLC, WHITEHAWK DELMAR LLC, WHITEHAWK D..."
2828426,2020,519 MEMORIAL 30312,220,"[KURTZ ZONDRA, HABITAT FOR HUMANITY IN ATLANTA..."


['34 PEACHTREE 2200 30303', '3495 PIEDMONT 11 30305', '3495 PIEDMONT 300 30305', '3505 KOGER 400 30096', '519 MEMORIAL 30312']
Tricon owned 1231 properties in 2020


,taxyr,owner_addr,count_owned_fulton_yr,assoc_owner_names
2652871,2020,1100 NORTHMEADOW 114 30076,180,"[CORE DEV GROUP LLC, OAKLEY TOWNSHIP HOMEOWNER..."
2652925,2020,1100 SPRING 550 30309,217,"[SELIG ENTERPRISES INC, SELIG ENTERPRISES INC,..."
2654673,2020,1110 NORTHCHASE 150 30067,424,"[NORTHWEST INTOWN DEVELOPMENT LLC, BROCK BUILT..."
2688099,2020,1508 BROOKHOLLOW 92705,252,"[TAH 2016 1 BORROWER LLC, TAH 2016 1 BORROWER ..."
2911711,2020,<NA> PO BOX 450233 31145,158,"[WILKINSON BEVERLY M & JERRY C, FR CAL THREE B..."


['1100 NORTHMEADOW 114 30076', '1100 SPRING 550 30309', '1110 NORTHCHASE 150 30067', '1508 BROOKHOLLOW 92705', '<NA> PO BOX 450233 31145']


## Create transcation scale table (includes purchases from govt + banks)

In [37]:
purchases_fulton_yr = pd.DataFrame(
    sales_grantee_dropped.groupby(["tax_year", "grantee_match_addr"])["parcel_id"].count().astype(int)
).reset_index().rename(columns={"parcel_id": "Purchases Fulton", "grantee_match_addr": "entity_addr"})

sales_fulton_yr = pd.DataFrame(
    sales_grantee_dropped.groupby(["tax_year", "grantor_match_addr"])["parcel_id"].count()
).reset_index().rename(columns={"parcel_id": "Sales Fulton", "grantor_match_addr": "entity_addr"})

sale_scale = purchases_fulton_yr.merge(
    sales_fulton_yr,
    on=["tax_year", "entity_addr"],
    how="outer"
)

sale_scale = sale_scale.fillna(0)
sale_scale["Purchases Fulton"] = sale_scale["Purchases Fulton"].astype(int)
sale_scale["Sales Fulton"] = sale_scale["Sales Fulton"].astype(int)
sale_scale["total_trans_fulton"] = sale_scale["Purchases Fulton"] + sale_scale["Sales Fulton"]

sale_scale.sort_values(by="total_trans_fulton", ascending=False).head(5)

,tax_year,entity_addr,Purchases Fulton,Sales Fulton,total_trans_fulton
377795,2022,<NA> PO BOX 4090 85261,540,88,628
196900,2018,5755 DUPREE 130 30327,65,548,613
361150,2022,4645 HAWTHORNE 20016,146,448,594
275428,2020,5001 PLAZA ON THE 200 78746,442,151,593
64199,2013,84 CHAIN LAKE 500 NAN,535,0,535


## Save owner and transcation scale

In [38]:
OUTPUT_PATH = 'output/'

owner_scale.to_csv(OUTPUT_PATH + 'owner_scale.csv', index=False)
sale_scale.to_csv(OUTPUT_PATH + 'sale_scale.csv', index=False)

## Create a valid sales only dataset

In [39]:
# Finish sales cleaning after dropping govt inst and banks
sales_full_valid = sales_full[sales_full["sale_val"] == "0"]
print(f"Number of valid sales: {len(sales_full_valid)}")

Number of valid sales: 183489


## Save

In [40]:
sales_full.to_csv(OUTPUT_PATH + 'sales_full_final.csv', index=False)
sales_full_valid.to_csv(OUTPUT_PATH + 'sales_full_valid.csv', index=False)
digest_full.to_csv(OUTPUT_PATH + 'fulton_digest_full_final.csv', index=False)